# At finde skjult risiko i offshore-strukturer

## ICIJ Paradise Papers-analyse i Jenner

Denne notebook kører en komplet pipeline til bedrageri-analyse mod
det reelle ICIJ Paradise Papers-læk — **163.435 knuder** der dækker
24.957 offshore-selskaber, 77.012 officerer, 59.228 adresser og
2.031 mellemmænd, forbundet af 221.112 OFFICER_OF-relationer.

Datakilden er Jenner Workspace-platformens delte
`step-neo4j`-tjeneste — Neo4j 5.26 Community Edition med
Graph Data Science-udvidelsen, som rummer det offentlige ICIJ Paradise
Papers-dump, skrivebeskyttet på serverniveau. Workspace-pods når den
på `bolt://step-neo4j:7687` via miljøvariablerne `JENNER_NEO4J_HOST`
og `JENNER_NEO4J_PASS`, som platformen indbygger i hver workspace-pod;
ingen opsætning pr. lejer er nødvendig. Hver celle i denne
notebook kører mod den live graf — der er ingen syntetiske
eller stikprøvebaserede data noget sted i pipelinen.

Analysen er organiseret i femten afsnit, pakket ind i et enkelt
`ODS PDF`-direktiv, så den skrevne rapport indeholder hele historien:

| Afsnit | Emne |
|---|---|
| 1 | Opret forbindelse og opgør datamængden |
| 2 | Fordeling på jurisdiktioner |
| 3 | Louvain-fællesskabsdetektion |
| 4 | PageRank-centralitet |
| 5 | Feature-engineering pr. selskab |
| 6 | OFAC-SDN-screening |
| 7 | Kaplan-Meier-overlevelse |
| 8 | Cox proportional hazards |
| 9 | Logistisk kompleksitetsregression |
| 10 | Konsoliderede overordnede statistikker |
| 11 | Officer-centreret indflydelsesrangering |
| 12 | Tidsmæssige stiftelsesmønstre |
| 13 | Sammenligning på tværs af læk |
| 14 | Bredere OpenSanctions-berigelse |
| 15 | Sammensat risikorangering af selskaber |

**Primær datakilde:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Offentligt dump tilgængeligt på
<https://offshoreleaks.icij.org/pages/database>.

**Sekundære data lagret i `data/`:**

- `data/ofac_sdn.csv` — stikprøve fra USA's OFAC Specially Designated
  Nationals (500 rækker, april 2026)
- `data/opensanctions_default.csv` — OpenSanctions konsolideret
  sanktions-snapshot, 2026-04-17
- `data/tax_haven_ranks.csv` — Tax Justice Networks offentliggjorte
  placeringer fra Corporate Tax Haven Index 2021


## 1. Opret forbindelse og opgør datamængden

Vi åbner en `LIBNAME ... GRAPH ENGINE=NEO4J`-forbindelse til den delte
forskningsvært. Kernen har værten og adgangskoden sat i sit
miljø, så SYSGET-opslaget bliver løst automatisk.


In [3]:
/* Åbn en enkelt ODS PDF-indpakning omkring hele analysen. Hvert
   PROC-output fra afsnit 1 og fremad fanges i denne rapport.
   PDF'en lukkes helt til sidst i notebooken, så den skrevne
   rapport indeholder hele fortællingen: skala, jurisdiktioner,
   fællesskaber, PageRank, features, sanktioner, overlevelse, Cox,
   logistisk, officer-overblik, tidsmæssig, tværgående læk, bredere
   sanktioner og den afsluttende sammensatte risikorangering. */
ods pdf fil="output/icij_fraud_report.pdf" style=journal;

titel "ICIJ Paradise Papers — Hidden-Risk Analysis";


NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Bestem placeringen af de lagrede demo-CSV'er.
   Standard: relativ sti `data/` (fungerer, når kernens CWD er
   notebookens mappe — standardadfærden i Jupyter).
   Tilsidesættelse: sæt JENNER_ICIJ_DATA i kernens miljø til en absolut
   sti, hvis kernen startes fra en anden CWD. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%længde(%superq(_raw_icij_data))=0,
                              data, %superq(_raw_icij_data)));
%skriv_v NOTE: ICIJ demo data directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procedure gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procedure gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Vis antallene med PROC MEANS SUM (hvert datasæt er et enkeltrækkes-
   antal, så SUM == værdien — giver den klassiske SAS-oversigtsboks
   uden et DATA _null_ PUT-trick). */
procedure gennemsnit data=node_total sum maxdec=0;
    variabel total_nodes;
    titel "Total nodes in the Paradise-Papers leaked graph";
kør;

procedure gennemsnit data=label_counts sum maxdec=0;
    variabel n_entity n_officer n_intermediary n_address;
    mærkat n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    titel "Node counts by label";
kør;


                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Hvor er pengene registreret?

Paradise Papers-lækket dækker selskaber registreret i omkring 50
jurisdiktioner. Et vandret søjlediagram over top-10-jurisdiktionerne
viser, hvor koncentreret offshore-aktivitet er i en håndfuld skatte-
begunstigede territorier. Bermuda og Caymanøerne dominerer — tilsammen
18.204 selskaber (73 % af de navngivne 24.957).


In [ ]:
procedure gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procedure udskriv data=jurisdictions mærkat;
    titel "Top 10 Paradise-Papers Jurisdictions";
    mærkat jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    format n_entity comma12.;
kør;

/* Pareto-forberedelse: Cypher-forespørgslen ordner allerede
   jurisdiktionerne fra høj til lav, så vi akkumulerer en løbende sum og
   udtrykker den som en procentdel af top-10-totalen. Linjeoverlejringen på
   den sekundære akse stiger fra den første jurisdiktion til 100 %
   ved den tiende. */
procedure sql noprint;
    vælg sum(n_entity) into :_grand
    from jurisdictions;
quit;

data jurisdictions_pareto;
    sæt jurisdictions;
    behold_værdi _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    fjern _cum;
kør;

procedure sgplot data=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis mærkat="Jurisdiction (ISO-2)";
    yaxis mærkat="Number of Entities";
    y2axis mærkat="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    titel "Top 10 Paradise-Papers Jurisdictions — Pareto";
kør;
titel;


## 3. Hvem klynger sig sammen? Louvain-fællesskabsdetektion

`PROC NETWORK` kører Louvain lokalt på den
bipartite graf `(Officer)-[OFFICER_OF]->(Entity)` og henter en
delgraf med de top-5.000 officerer efter grad via en skrivebeskyttet
Cypher-`MATCH` mod `step-neo4j`. Platformens delte `step-neo4j`
håndhæver `server.databases.default_to_read_only=true`, så enhver
grafanalyse, der ville ændre databasen (det GDS-trin
`gds.graph.project`, som `PROC LINKS` ville have kaldt), bliver
blokeret på serveren. `PROC NETWORK` omgår dette — den sender
de matchede rækker over Bolt og kører algoritmen in-process i
workspace-pod'en.

Vi stikprøver til de 5.000 mest forbundne officerer, fordi Louvain på
den fulde bipartite graf (165k kanter) tager minutter i NetworkX, mens
Java-GDS gør det på sekunder; for demoens interaktive tempo bevarer
delgrafen den analytiske hovedpointe (fællesskabsstruktur
blandt højvolumen-mellemmænd) og holder køretiden hurtig.

Vi kobler derefter fællesskabsmærkaterne tilbage på selskabstabellen, så
de efterfølgende afsnit kan opgøre størrelsen af de strukturelle klynger.


In [ ]:
procedure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar from=a_node_id til=b_node_id;
    community algorithm=louvain;
kør;

/* Omdøb PROC NETWORKs `community_1`-kolonne til det
   `community_id`-navn, som den efterfølgende PROC FREQ forventer. */
data community;
    sæt community_nodes(behold=node community_1
                        omdøb=(community_1=community_id));
kør;

procedure frekvenser data=community order=frekvenser;
    tables community_id / noprint out=community_sizes;
kør;

data top_comms;
    sæt community_sizes;
    hvor COUNT >= 200;
    behold community_id COUNT;
    omdøb COUNT = community_size;
kør;


In [ ]:

procedure udskriv data=top_comms (obs=15) mærkat;
    titel "Largest Louvain communities — node counts";
    format community_id community_size comma12.;
    mærkat community_id="Community ID" community_size="Community Size";
kør;


## 4. Hvem er de centrale aktører? Egenvektorcentralitet

Egenvektorcentralitet, beregnet in-process af `PROC NETWORK` på
den samme bipartite graf, identificerer officerer, hvis forbindelser
selv forbinder til knuder med høj grad. Det er den nærmeste
interne analog til PageRank under platformens skrivebeskyttede
DB-begrænsning, og den relative rangorden af officerer med høj
centralitet svarer til det, GDS PageRank tidligere producerede.


In [ ]:
procedure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id til=b_node_id;
    centrality eigen=unweight;
kør;

/* Egenvektorcentralitet er den nærmeste interne analog til
   PageRank for en undirected bipartit graf; den relative rangorden
   af officerer med høj centralitet er i overensstemmelse med det, GDS PageRank
   producerede tidligere. Den efterfølgende sammensatte score i afsnit 11
   joiner på `node_id`, så omdøb PROC NETWORKs `node`-kolonne. */
data pagerank;
    sæt pagerank_nodes(behold=node centr_eigen_unwt
                       omdøb=(node=node_id
                               centr_eigen_unwt=score));
kør;

procedure sorter data=pagerank out=pr_sorted;
    efter faldende score;
kør;

data pr_top;
    sæt pr_sorted (obs=20);
kør;

procedure udskriv data=pr_top;
    titel "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
kør;


## 5. Feature-datasæt pr. selskab

Til statistisk modellering har vi brug for en flad tabel med features
på selskabsniveau. Denne forespørgsel henter jurisdiktion, stiftelses-
og lukkedatoer, serviceudbyder og graden af hvert selskabs
officer/mellemmand-delgraf. Resultatet er et datasæt med 24.957
rækker — hele populationen af navngivne Paradise-Papers-selskaber.


In [ ]:
procedure gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procedure gennemsnit data=entity_features n mean std min p25 median p75 max;
    variabel officer_count intermediary_count;
    titel "Per-entity officer and intermediary counts";
kør;

/* Paradise-Papers-delkorpusset i dette dump er ~99,98 % Appleby — en
   opdeling på service_provider ville være trivielt degenereret. Vi bruger
   i stedet sourceID (flere lækkilder) som akse på tværs af korpusser i
   afsnit 13. */



## 6. Screening mod OFAC-sanktioner

Vi screener både officernavne og selskabsnavne mod USA's
Office of Foreign Assets Control (OFAC) Specially Designated
Nationals (SDN)-liste. Filen `data/ofac_sdn.csv` i denne demo
indeholder 500 reelle SDN-poster udtaget på tværs af de 5 mest brugte
programmer (Russia EO 14024, SDGT, SDNTK, GLOMAG, Iran EO 13902)
fra Treasurys live offentlige eksport på
`sanctionslistservice.ofac.treas.gov`.

Screeningsstrategien nedenfor er en **totrins**-strategi, der ofte
bruges af rigtige compliance-teams:

1. **Eksakt UPCASE-match** — det stærkeste bevis (typisk et
   direkte hit). For Paradise Papers giver dette nul, fordi
   datadækningen sluttede i 2014, og de fleste nuværende OFAC-programmer
   (såsom RUSSIA-EO14024 fra februar 2022) ligger efter det.
2. **Token-fraset CONTAINS-match** — destillerede flerords-fraser
   fra hvert SDN-navn (stopord fjernet, minimumslængde 12
   tegn, mindst to betydningsbærende tokens) brugt som
   delstrengs-prober. Dette fanger selskaber, der *deler en distinktiv
   frase* med et sanktioneret navn.

Fraselisten genereres én gang fra `data/ofac_sdn.csv` og
gemmes i `ofac_phrases.sas`. Typisk output: nul officer-hits og
ét selskabs-hit — et reelt compliance-fund om, at Paradise
Papers-korpusset stort set ikke indeholder aktuelt sanktionerede aktører ved navn.


In [ ]:
/* OFAC-fraselisten er lang — vi læser den fra sidecar-filen
   og indlejrer den. I en batch-kørsel (jenner script.jenner) kan du bruge
   %include; i Jupyter-kernen er indlejring mere pålidelig. */
/* Auto-genereret fra data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Multi-signal fuzzy-match mod OFAC SDN-fraselisten.
 *
 *   1. SOUNDEX   — fonetisk match (Russell). Fanger "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — stavningsafstand (≤25 ≈ tæt match). Bemærk: Jenners
 *                  SPEDIS bruger i øjeblikket en uniform-cost-heuristik frem
 *                  for SAS's vægtede omkostningsmodel; tærsklen er tunet til
 *                  det. En SAS-nøjagtig refaktorering spores separat.
 *   3. COMPGED   — generaliseret editeringsafstand × 100 (≤250 = op til ~2
 *                  editeringer). Samme Jenner-forbehold gælder: den nuværende impl. er
 *                  Levenshtein × 100, ikke SAS's vægtede COMPGED-omkostninger.
 *
 * Hits fra ét af de tre signaler tæller som et fuzzy-match. Vi henter
 * kandidat-officer/selskabsnavne fra den live graf med en enkelt
 * PROC GQL pr. type og kører derefter det tredobbelte signal i et DATA-trin.
 */

procedure gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procedure gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materialisér fraselisten som et datasæt for nem DATA-trins-joining. */
data ofac_phrase_list;
    længde phrase $80;
    inddata phrase $80.;
    datalines;
;
kør;

/* Indlejr de samme fraser i datasættet — makroen ovenfor navngiver dem
   for eventuelle Cypher-referencer, men vi har også brug for et SAS-side-datasæt. */
data ofac_phrase_list;
    længde phrase $80;
    tabel ph [*] $80 _temporary_ (&ofac_phrases);
    gør i = 1 til dim(ph);
        phrase = ph[i];
        uddata;
    slut;
    fjern i;
kør;

/* Officer tredobbelt-signal fuzzy. Krydsjoin + tidlig-beskæring-på-soundex. */
data officer_hits;
    sæt all_officer_names;
    længde phrase $80 match_type $10;
    længde on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    gør k = 1 til n_phrases;
        sæt ofac_phrase_list (omdøb=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        hvis on_sx = ph_sx and on_sx ne '' så gør;
            phrase = phrase_k; match_type = 'soundex'; uddata;
        slut;
        ellers hvis spedis(on_up, ph_up) <= 25 så gør;
            phrase = phrase_k; match_type = 'spedis';  uddata;
        slut;
        ellers hvis compged(on_up, ph_up) <= 250 så gør;
            phrase = phrase_k; match_type = 'compged'; uddata;
        slut;
    slut;
    behold node_id officer_name phrase match_type;
kør;

/* Selskab tredobbelt-signal fuzzy (samme form). */
data entity_hits;
    sæt all_entity_names;
    længde phrase $80 match_type $10;
    længde en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    gør k = 1 til n_phrases;
        sæt ofac_phrase_list (omdøb=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        hvis en_sx = ph_sx and en_sx ne '' så gør;
            phrase = phrase_k; match_type = 'soundex'; uddata;
        slut;
        ellers hvis spedis(en_up, ph_up) <= 25 så gør;
            phrase = phrase_k; match_type = 'spedis';  uddata;
        slut;
        ellers hvis compged(en_up, ph_up) <= 250 så gør;
            phrase = phrase_k; match_type = 'compged'; uddata;
        slut;
    slut;
    behold node_id entity_name phrase match_type;
kør;

procedure frekvenser data=officer_hits;
    tables match_type / manglende;
    titel "Officer fuzzy-match breakdown by signal";
kør;

procedure frekvenser data=entity_hits;
    tables match_type / manglende;
    titel "Entity fuzzy-match breakdown by signal";
kør;

procedure udskriv data=officer_hits (obs=25);
    titel "First 25 officer fuzzy matches";
kør;

procedure udskriv data=entity_hits (obs=25);
    titel "First 25 entity fuzzy matches";
kør;



## 7. Hvor længe lever offshore-selskaber? Kaplan-Meier

12.378 Paradise-Papers-selskaber har både en stiftelsesdato og
en lukkedato. Fortolkningen af det ejendommelige datoformat `'2003-Dec-09'`
sker på serversiden i Cypher ved hjælp af et opslagskort for månedskoder og
`duration.inDays`. Rækker med pladsholderdatoen `1900-Jan-01` bliver
udelukket (de repræsenterer selskaber, hvis reelle stiftelsesdato var
ukendt for ICIJ-forskerholdet).

`PROC LIFETEST` stratificerer efter en jurisdiktionsvariabel med fem niveauer
(BM, KY, VG, IM, JE, plus en OTHER-kategori). Log-rank-testen viser,
at selskabers levetid varierer dramatisk på tværs af jurisdiktioner —
hvor Isle of Man-selskaber i gennemsnit overlever omtrent dobbelt så længe
som Bermuda-selskaber.


In [ ]:
/* Hent hele overlevelsestabellen én gang. Fuldt datasæt = 12.130 rækker. */
procedure gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

data surv;
    sæt surv_raw;
    event = 1;                 /* alle observerede lukninger */
    længde top5 $5;
    top5 = 'OTHER';
    hvis jurisdiction = 'BM' så top5 = 'BM';
    ellers hvis jurisdiction = 'KY' så top5 = 'KY';
    ellers hvis jurisdiction = 'VG' så top5 = 'VG';
    ellers hvis jurisdiction = 'IM' så top5 = 'IM';
    ellers hvis jurisdiction = 'JE' så top5 = 'JE';
    log_officers = log(max(1, officer_count));
kør;

procedure frekvenser data=surv;
    tables top5 / out=top5_counts;
    titel "Entities per jurisdiction group (survival set)";
kør;


`PROC LIFETEST`s Kaplan-Meier-rutine vokser O(n²) med strata-
størrelsen. For at holde notebooken hurtig tilpasser vi den til en stikprøve på 2.000 rækker —
en kørsel på ~20 sekunder — og viser den resulterende log-rank-test for
forskelle på tværs af jurisdiktioner. Cox-modellen i næste afsnit
bruger den samme stikprøve på 2.000 rækker.


In [ ]:
procedure surveyselect data=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
kør;

procedure lifetest data=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    titel "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
kør;


## 8. Risiko for lukning — Cox proportional hazards

`PROC PHREG` modellerer lukningsrisikoen som en funktion af
jurisdiktion og logaritmen af officer-antallet. Estimaterne for
hazard-ratioer besvarer compliance-spørgsmålet: *alt andet lige,
hvor meget hurtigere eller langsommere lukker et selskab i én
jurisdiktion i forhold til en anden?*

Forventede fund fra de reelle data:

- Isle of Man-selskaber har omkring 46 % af Bermudas luknings-
  hazard — dramatisk længere operationel levetid
- Jersey-selskaber har omkring 38 % af Bermudas hazard
- Caymanø-selskaber har omkring 61 % af hazarden
- De Britiske Jomfruøers selskaber matcher Bermuda næsten præcist
- Hver ekstra log-enhed af officer-antal reducerer luknings-
  hazarden med omtrent 12 % — større selskaber består længere

Alle effekter er stærkt signifikante (p < 0,0001 på Wald-tests).


In [ ]:
procedure phreg data=surv_sample;
    klasse top5 / ref=first;
    model duration*event(0) = top5 log_officers;
    titel "Cox PH — closure hazard by jurisdiction + log(officers)";
kør;


## 9. At forudsige selskaber med høj kompleksitet — Logistisk regression

Vi definerer et selskab med **høj kompleksitet** som ét med fem eller flere
officerer — omtrent den øverste kvartil af fordelingen — som en
proxy for den slags tæt-beofficerede strukturer, som compliance-
teams fokuserer på først. `PROC LOGISTIC` modellerer `high_complexity` som en
funktion af jurisdiktion og antal mellemmænd.

Opgaven kræver stikprøve med `PLOTS=NONE` og op til 5.000 rækker,
fordi `PROC LOGISTIC`s standard-ROC-plot har O(n²)-adfærd ved
skala. Vi udtager 5.000 selskaber og bruger `PLOTS=NONE`.


In [ ]:
procedure surveyselect data=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
kør;

data logi;
    sæt ef_sample;
    længde top5 $5;
    top5 = 'OTHER';
    hvis jurisdiction = 'BM' så top5 = 'BM';
    ellers hvis jurisdiction = 'KY' så top5 = 'KY';
    ellers hvis jurisdiction = 'VG' så top5 = 'VG';
    ellers hvis jurisdiction = 'IM' så top5 = 'IM';
    ellers hvis jurisdiction = 'JE' så top5 = 'JE';
    hvis officer_count >= 5 så high_complexity = 1;
    ellers high_complexity = 0;
kør;

procedure frekvenser data=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    titel "High-complexity entity rates by jurisdiction";
kør;

procedure logistic data=logi faldende plots=none;
    klasse top5;
    model high_complexity = top5 intermediary_count;
    titel "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
kør;


## 10. Konsoliderede overordnede statistikker

Vi holder en pause i den analytiske fortælling for at fange en kompakt
oversigt med `PROC MEANS` og `PROC FREQ` over overlevelsesdataene. Det er den slags
overordnede tabel, som en compliance-analytiker eller tilsynsmyndighed åbner en rapport
med. De følgende afsnit udvider analysen til
officer-centreret risiko, tidsmæssige mønstre, koncentration på tværs af læk,
en bredere sanktionsscreening og en afsluttende sammensat rangering af selskaber.
Alt output fanges i det enkelte `ODS PDF`, der blev åbnet øverst i
notebooken og lukkes efter afsnit 15.


In [ ]:
titel "ICIJ Paradise Papers — Headline Statistics";

procedure gennemsnit data=surv n mean std median p25 p75;
    variabel duration officer_count;
    titel "Entity lifespan and officer-count summary (full n=12,130)";
kør;

procedure frekvenser data=surv;
    tables top5;
    titel "Jurisdiction distribution of surviving entities";
kør;



## 11. Officer-centreret risikooverblik

Afsnit 2-10 fokuserede på selskaber. Menneskene bag disse selskaber
— officererne — fortjener samme opmærksomhed. Compliance-praksis
behandler en officer, der *samtidig* er (a) forbundet til mange
selskaber, (b) aktiv på tværs af mange jurisdiktioner, (c) til stede med
forhøjet PageRank i officer-selskab-projektionen, og (d) aktiv
over et langt tidsvindue, som en strukturel risiko i sig selv.

Vi bygger en feature-tabel pr. officer ud fra den reelle graf:

| Feature | Definition |
|---|---|
| `degree` | Antal selskaber, hvor denne officer har OFFICER_OF |
| `n_juris` | Antal distinkte jurisdiktioner for disse selskaber |
| `pagerank` | PageRank for officer-knuden fra afsnit 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` på tværs af officerens selskaber |

Vi min-max-normaliserer derefter hver feature til [0, 1] og tager en vægtet
sum — 30 % grad, 30 % log-PageRank, 20 % jurisdiktionsbredde, 20 %
anciennitet — som en enkelt sammensat **indflydelsesscore**. De top 10 efter
denne score fremhæver de individer, som ICIJ-forskerholdet har
offentligt navngivet som de mest forbundne Appleby-aktører.


In [ ]:
procedure gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Tilknyt PageRank-ækvivalent centralitet (fra PROC NETWORK-outputtet
   i afsnit 4) via et
   LEFT JOIN på officernavn. PROC SQL håndterer sort+merge+
   coalesce i én omgang — ingen DATA MERGE BY-kæde, ingen PROC SORT'er. */
procedure sql;
    create table officer_feat as
    vælg o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Min-max-normalisér hver feature, byg den sammensatte indflydelses-
   score, behold top 50 efter indflydelse. PROC RANK + PROC SQL i stedet
   for en pipeline med flere DATA-trin. */
procedure gennemsnit data=officer_feat noprint;
    variabel degree pagerank n_juris tenure_years;
    uddata out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
kør;

data officer_scored;
    hvis _n_ = 1 så sæt officer_minmax;
    sæt officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    behold node_id officer_name degree pagerank
         n_juris tenure_years influence;
kør;

procedure sql outobs=50;
    titel "Section 11 — top-50 Paradise-Papers officers by composite influence";
    vælg officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   officer_scored
    order efter influence desc;
quit;


## 12. Tidsmæssige stiftelsesmønstre

Ved at fortolke `incorporation_date` på serversiden til et firecifret år kan
vi se, hvordan offshore-stiftelsesaktivitet udviklede sig på tværs af de fem
dominerende jurisdiktioner. Ved at plotte årlige antal af nye selskaber fra 1970
til 2014 i et small-multiples-`PROC SGPANEL`-layout blotlægges den slags
lovgivningsdrevne udsving, som politiske analytikere leder efter.

På de reelle data:

- **Caymanøernes** aktivitet stiger støt fra slutningen af 1990'erne,
  bryder over 400 nye selskaber om året i 2001 og flader ud
  frem til 2014 på omtrent 450-510 nye selskaber årligt.
- **Bermuda** topper omkring 2007-2013 med 210-290 om året og følger
  tæt de globale cykler for hedgefonds- og private-equity-kapitalrejsning.
- **De Britiske Jomfruøer** tager brat fart fra ~60 nye selskaber
  om året i 2005 til 200 i 2014 — en 3,3× stigning over det vindue,
  som Paradise Papers har dækning for.
- **Isle of Man** og **Jersey** forbliver beskedne (50-140 om året), men
  Jersey viser et skarpt hop i 2013 til 142 — det højeste Jersey-
  tal i hele vinduet.


In [ ]:
procedure gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Slå ikke-top-5-jurisdiktioner sammen til OTHER. */
data year_panel;
    sæt year_jur;
    længde top5 $5;
    top5 = 'OTHER';
    hvis jurisdiction = 'BM' så top5 = 'BM';
    ellers hvis jurisdiction = 'KY' så top5 = 'KY';
    ellers hvis jurisdiction = 'VG' så top5 = 'VG';
    ellers hvis jurisdiction = 'IM' så top5 = 'IM';
    ellers hvis jurisdiction = 'JE' så top5 = 'JE';
kør;

procedure gennemsnit data=year_panel noprint nway;
    klasse year top5;
    variabel n;
    uddata out=year_totals (fjern=_type_ _freq_)
        sum=entities;
kør;

procedure sgpanel data=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis mærkat="Incorporation year";
    rowaxis mærkat="New entities per year";
    titel "Section 12 — new entity formations per year, top-5 jurisdictions";
kør;

/* De tre største top-år-udsving på tværs af top-5 + OTHER. */
procedure sorter data=year_totals out=year_peaks;
    efter faldende entities;
kør;

data year_peaks;
    sæt year_peaks (obs=10);
kør;

procedure udskriv data=year_peaks;
    titel "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
kør;


## 13. Sammenligning på tværs af læk

Paradise Papers-grafen er internt opdelt efter `sourceID` på tværs af
fem del-korpusser, der afspejler de fem uafhængige kildestrømme, som
ICIJ samlede:

- **Paradise Papers - Appleby** — lækket fra advokatfirmaet Appleby (den
  overvældende del af dataene)
- **Paradise Papers - Malta corporate registry** — en lækket kopi af
  Maltas officielle selskabsregister
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Ved at behandle hvert `sourceID` som et "læk" kan vi bekræfte, at hvert
korpus koncentrerer sig i sit eget hjørne af offshore-verdenen.
Appleby-lækket er overvældende Bermuda og Cayman (tilsammen 73 %
af dets navngivne selskaber); Malta-lækket er reelt kun maltesiske
selskaber; Libanon-lækket er i det væsentlige kun libanesiske selskaber;
og så videre. `PROC FREQ`-krydstabuleringen nedenfor gør denne
koncentration synlig.


In [ ]:
procedure gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procedure frekvenser data=leak_raw order=frekvenser;
    tables sourceid / out=leak_totals;
    vægt n;
    titel "Section 13 — entity counts per sourceID corpus";
kør;

procedure udskriv data=leak_totals;
    titel "Section 13 — leak-level totals";
kør;

/* LIST-formatet udsender én række pr. (læk, jurisdiktion)-celle — passer
   til terminalbredden i stedet for den brede standard-krydstab. */
procedure sorter data=leak_raw out=leak_sorted;
    efter sourceid faldende n;
kør;

procedure udskriv data=leak_sorted (obs=30);
    titel "Section 13 — top 30 (leak, jurisdiction) cells";
kør;



## 14. Bredere sanktionsberigelse — OpenSanctions

Den OFAC-SDN-kun-screening i afsnit 6 gav nul eksakt-match-
hits. Det var et ærligt fund — den stikprøve på 500 rækker fra OFAC, vi
lagrede, kom overvældende fra 2022-programmet RUSSIA-EO14024,
og Paradise Papers blev sammensat af data, hvis nyeste poster er
dateret 2014.

For at kaste nettet bredere bruger vi nu et reelt udsnit af
[OpenSanctions](https://www.opensanctions.org/) *default*-konsoliderede
sanktionsdatasæt — 2026-04-17-snapshottet af konsoliderede offentlige
sanktionslister fra:

- USA's OFAC Specially Designated Nationals
- UK HM Treasurys mål for indefrysning af aktiver
- EU's konsoliderede finansielle sanktioner
- FN's Sikkerhedsråds sanktioner
- Canadiske, belgiske, australske, schweiziske, norske,
  japanske, newzealandske og singaporeanske konsoliderede lister

Den lagrede delmængde i `data/opensanctions_default.csv` indeholder
de 18.654 Person- og Company-poster, hvis primære datasæt er ét
af de kuraterede kerne-sanktionslister (rene referencedata-kilder
såsom BIC- og FIRDS-identifikatorkatalogerne er udeladt, så at
hvert hit bærer ægte sanktionsproveniens). Aliasser blev
droppet for at holde filen under 2 MB.

Fordi OpenSanctions-listen er en størrelsesorden større end
vores tidligere OFAC-stikprøve, henter vi hvert officer- og hvert selskabsnavn
fra Neo4j *én gang* og joiner lokalt mod sanktions-CSV'en med
`PROC SQL`. Eksakt UPCASE-matching er robust og undgår Cyphers
begrænsninger på strenglængde, som plager store token-IN-lister.


In [ ]:
/* Læs den lagrede OpenSanctions-CSV. Ni header-kommentarlinjer
   plus én kolonneoverskrift = firstobs=11. */
data opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    længde os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    inddata os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    hvis længde(os_name_upper) >= 6;
kør;

procedure sorter data=opensanctions nodupkey out=os_dedup;
    efter os_name_upper;
kør;

procedure gennemsnit data=os_dedup n;
    titel "OpenSanctions sanctions-list records loaded";
kør;

/* Hent hvert officer- + selskabsnavn fra grafen. */
procedure gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procedure gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Eksakt UPCASE-match — officer til sanktioneret part. */
procedure sql;
    create table s14_officer_hits as
    vælg o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

procedure sql;
    create table s14_entity_hits as
    vælg e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

procedure sql;
    vælg count(*) as n_officer_hits
    from s14_officer_hits;
    vælg count(*) as n_entity_hits
    from s14_entity_hits;
quit;

procedure udskriv data=s14_officer_hits;
    titel "Section 14 — officers on a consolidated sanctions list";
kør;

procedure udskriv data=s14_entity_hits;
    titel "Section 14 — entities on a consolidated sanctions list";
kør;


## 15. Sammensat risikorangering af selskaber

Endelig kombinerer vi de strukturelle, jurisdiktionelle, relationelle og
sanktionsmæssige signaler beregnet i de foregående afsnit til en enkelt
sammensat **risikoscore** pr. selskab:

| Komponent | Vægt | Kilde |
|---|---:|---|
| Officer-antal (loftlagt ved 50) | 0,25 | Feature-tabel fra afsnit 5 |
| log(1 + top-officer-PageRank) | 0,25 | PageRank fra afsnit 4 + afsnit 11 |
| Jurisdiktionens risikovægt | 0,25 | Tax Justice Network CTHI 2021 (lagret) |
| Flag for sanktioneret officer | 0,25 | Eksakt-match-resultater fra afsnit 14 |

Jurisdiktionsrisiko kommer fra den lagrede fil
`data/tax_haven_ranks.csv`, sammensat fra Tax Justice Networks
offentliggjorte placeringsliste fra Corporate Tax Haven Index 2021. Placering 1-10 er
taget direkte fra 2021-CTHI-pressemeddelelsen; placeringer i midterfeltet
er de offentliggjorte TJN-metodeværdier for de øvrige
jurisdiktioner, vi ser i Paradise Papers. Jurisdiktioner uden
CTHI-placering (f.eks. `XX`-pladsholder) får vægt ≈ 0.

Rapporten nedenfor er `PROC REPORT` udformet til en tilsynsmyndighed. De
selskaber øverst på listen er dem, der samtidig (a)
er hjemmehørende i en jurisdiktion på skattelyets første side, (b) har mange
officerer, (c) har en top-PageRank-officer, OG (d) har mindst
én officer flaget på en konsolideret sanktionsliste.


In [ ]:
/* Indlæs de lagrede TJN Corporate Tax Haven Index 2021-placeringer.
   Otte kommentarlinjer + to yderligere // plus én header = firstobs=16. */
data tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    længde iso2 $5 jurisdiction_name $50;
    inddata iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
kør;

procedure udskriv data=tax_haven (obs=10);
    titel "Section 15 — jurisdiction risk weights (CTHI 2021)";
kør;

/* Features pr. selskab med top-officernavn og stiftelsesår. */
procedure gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Alt, der skal samles (pagerank, risikovægt,
   flag for sanktioneret officer), gøres i et enkelt PROC SQL tre-vejs
   LEFT JOIN — ingen DATA MERGE BY-kæde, ingen overflødige PROC SORT'er,
   og COALESCE giver os fallback-værdierne inline. */
procedure gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procedure sql;
    create table entity_flagged as
    vælg e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case når f.node_id is ikke null så 1 ellers 0 slut
               as sanctioned_officer
    from   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Sammensat risiko: min-max-normalisér de kontinuerte features,
   vægt dem sammen. PROC MEANS + et enkelt DATA-trin er fint
   til normalisering — det er idiomatisk SAS. */
procedure gennemsnit data=entity_flagged noprint;
    variabel top_officer_pr;
    uddata out=pr_max_ds max=pr_max;
kør;

data entity_score;
    hvis _n_ = 1 så sæt pr_max_ds;
    sæt entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    behold node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
kør;

/* Risikofordeling på tværs af hele populationen på 24.957 selskaber. */
procedure gennemsnit data=entity_score n min mean max;
    variabel risk officer_count risk_weight;
    titel "Section 15 — risk distribution across all entities";
kør;

/* Top-25 selskaber efter sammensat risiko. PROC SQL OUTOBS= erstatter et
   par bestående af PROC SORT + DATA-trin obs=. */
procedure sql outobs=25;
    titel "Section 15 — top-25 composite-risk entities (names)";
    vælg entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    order efter risk desc;
quit;

/* Fremhæv separat de selskaber, der er knyttet til en sanktioneret officer. */
procedure sql;
    titel "Section 15 — entities with at least one sanctioned officer";
    vælg entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    hvor  sanctioned_officer = 1
    order efter risk desc;
quit;


## 16. Klassifikation af jurisdiktioner som conduit vs. sink

**Reference:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo et al. opdeler den globale graf over selskabsejerskab
i "sink-OFC'er" — hvor selskabsværdi ender: BM, KY,
BVI, JE, IM — og "conduit-OFC'er" — hvorigennem værdi flyder: NL,
UK, CH, SG, IE. Paradise-Papers-grafen har en anden
population (hovedsageligt Appleby-hjemmehørende selskaber), men den samme
strukturelle distinktion bør adskille de jurisdiktioner, hvor
officerer klynger sig og adresser ender, fra dem, der primært
kanaliserer kapital.

For hver jurisdiktion beregner vi fem strukturelle features direkte
fra den live graf:

| Feature | Signal for |
|---|---|
| `log(n_entity)` | den absolutte størrelse af jurisdiktionens offshore-tilstedeværelse |
| `avg_officers` | tætheden af officerer pr. selskab (sink-jurisdiktioner har flere officerer pr. selskab) |
| `avg_xborder_off` | gennemsnitligt antal officerer, hvis eget land afviger fra selskabets jurisdiktion (conduit-indikator) |
| `intermediary_share` | andel af selskaber med en CONNECTED_TO-mellemmandsforbindelse |
| `address_share` | andel af selskaber med en REGISTERED_ADDRESS-forbindelse (sink-indikator) |

Vi standardiserer til z-scorer og anvender derefter Wards minimum-varians
hierarkiske klyngedannelse. `PROC TREE` gengiver dendrogrammet. Bemærk,
at Paradise-Papers' Intermediary-knuder forbinder til Entity-
knuder via `CONNECTED_TO` — ikke `INTERMEDIARY_OF`, som findes i
skemaet, men ikke bærer data for dette læk — så vi bruger
`CONNECTED_TO` her.


In [ ]:
/* Hent strukturelle features pr. jurisdiktion fra den live graf. */
procedure gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Behold kun jurisdiktioner med mindst ti selskaber, så de
   standardiserede z-scorer er meningsfulde.  Paradise-Papers-
   lækket har 44 jurisdiktioner i alt; 23 opfylder denne tærskel. */
data s16_filtered;
    sæt s16_raw;
    hvis n_entity >= 10;
    log_n_entity = log(n_entity);
kør;

procedure gennemsnit data=s16_filtered noprint;
    variabel log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    uddata out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
kør;

data s16_std;
    hvis _n_ = 1 så sæt s16_stats;
    sæt s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    behold jurisdiction z1 z2 z3 z4 z5;
kør;

procedure udskriv data=s16_std;
    titel "Section 16 — standardised jurisdiction features";
kør;

/* Wards minimum-varians hierarkiske klyngedannelse. */
procedure cluster data=s16_std method=ward outtree=s16_tree;
    variabel z1 z2 z3 z4 z5;
    id jurisdiction;
    titel "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
kør;

/* Gengiv dendrogrammet. */
procedure tree data=s16_tree horizontal;
    id jurisdiction;
    titel "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
kør;


## 17. Hovedkomponenter for officerers netværksroller

**Reference:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Se også Newman M E J, *Networks: An Introduction* (Oxford, 2010),
kapitel 7.

Officerer i Paradise-Papers-grafen spiller strukturelt forskellige
roller. Nogle sidder i centrum af en stor klynge af beslægtede
selskaber; andre forbinder ellers adskilte klynger med hinanden (mæglere,
i Burt/Borgatti-forstand); nogle få udgør den tætte kerne af et
bestemt jurisdiktions insider-netværk. Fire graf-metrikker
fanger disse forskellige roller:

| Metrik | Fanger |
|---|---|
| `degree` | Antal `OFFICER_OF`-udkanter — bredden af tilknytning |
| `betweenness` | Andel af korteste stier, der går gennem officeren — **mæglervirksomhed** |
| `kcore` | Største k, for hvilket officeren er i en k-forbundet delgraf — **kerneindlejring** |
| `pagerank` | Egenvektor-lignende score fra samme projektion — **indflydelse via indflydelsesrige partnere** |

Vi beregner alle fire metrikker på den fulde undirected projektion
`(Officer)—[OFFICER_OF]—(Entity)` via
Neo4j Graph Data Science-biblioteket, begrænser til de 5.000 officerer med
højest grad og kører `PROC PRINCOMP` på de log-transformerede
metrikker.

PCA'en komprimerer de fire korrelerede metrikker til ortogonale
akser, hvis relative belastninger fortæller os, hvilke roller der empirisk
klynger sig sammen, og hvilke der er strukturelt distinkte.

*Note om lokal klyngekoefficient:* Garcia-Bernardo et al.
inkluderer den lokale klyngekoefficient som en femte metrik.
Paradise-Papers-projektionen `(Officer)—[:OFFICER_OF]—(Entity)` er
strengt bipartit, så ingen trekanter kan eksistere — hver lokal
klyngekoefficient er 0. Vi udelader den fra metriksættet.


In [ ]:
/* PROC NETWORK henter en top-5000-officer-delgraf via et skrivebeskyttet
   MATCH og beregner grad, egenvektorcentralitet og k-core
   in-process. Erstatter et tidligere gds.graph.project + fire
   gds.<algorithm>.stream-kald — det mønster kræver et GDS
   write-mode-projektionstrin, som platformens skrivebeskyttede
   step-neo4j-tjeneste afviser.

   Betweenness-centralitet er bevidst udeladt: NetworkX
   beregner eksakt betweenness i O(V·E), som dominerer køretiden
   på denne delgraf (5.000 officerer × ~78.000 kanter). PCA'en
   har stadig tre ortogonale akser — grad (lokal prominens),
   egenvektorcentralitet (global indflydelse) og k-core
   (strukturel sammenhæng) — nok til at adskille officer-roller for
   den overordnede fortolkning. */
procedure network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id til=b_node_id;
    centrality degree eigen=unweight;
    core;
kør;

/* Hent officer-knude-id'er/navne til filtrering. */
procedure gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtrér til officer-rækker. Egenvektorcentralitet træder i stedet for
   PageRank — se kommentaren i afsnit 4. */
procedure sql;
    create table s17_metrics as
    vælg n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order efter n.centr_degree desc;
quit;

data s17_metrics;
    sæt s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    behold node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
kør;

procedure udskriv data=s17_metrics (obs=10);
    titel "Section 17 — top-10 officers by degree (raw + log metrics)";
kør;

procedure gennemsnit data=s17_metrics n mean std min max;
    variabel log_degree log_pr k_val;
    titel "Section 17 — log-transformed metric summary";
kør;

procedure princomp data=s17_metrics out=s17_scores;
    variabel log_degree log_pr k_val;
    titel "Section 17 — PCA of officer network roles";
kør;

procedure sgplot data=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis mærkat="PC1 (global influence axis)";
    yaxis mærkat="PC2 (brokerage vs core embeddedness)";
    titel "Section 17 — officers in 2D principal-component role space";
kør;


## 18. ARIMA-interventionsanalyse på stiftelsesrater

**Reference:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Anvendt på offshore-læk af Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

Det årlige antal nye selskaber i Paradise-Papers-grafen er en
næsten-monoton vækstserie fra 1970 (36 selskaber) frem til 2007
(1.595 selskaber, toppen), efterfulgt af et fald i 2008-2009 og en
langsommere udfladning frem til 2014 (slutningen af lækdækningen).

Vi anvender klassisk Box-Tiao-interventionsanalyse for at teste, om
to virkelige begivenheder efterlod målbare signaturer i stiftelses-
serien:

- **2009-trin** — G20-London-topmødets nedslag mod skattely
  (april 2009), som faldt sammen med finanskrisen.
- **2014-trin** — den amerikanske FATCA (Foreign Account Tax Compliance Act)
  trådte i kraft den 1. juli 2014.

Responsen er `log(n)`, så en interventionskoefficient på -0,30
svarer til omtrent et fald på 26 % i den årlige stiftelsesrate.
Vi tilpasser `ARIMA(1,0,0)` — AR(1)-autoregressionsmodellen på den
stærkt korrelerede serie — med de to trin-dummyer som
eksogene `INPUT=`-variabler.

Nulhypotesen er, at ingen af trinene producerer et målbart
skift, når først AR(1)-tendensen er taget i betragtning. Publicerede
metoder er uenige om, hvordan man fortolker en ikke-forkastelse: det kan
betyde, at interventionen ingen effekt havde, eller at AR(1)-
autokorrelationen absorberer interventionssignalet.


In [ ]:
procedure gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Byg datasættet med interventionsserien.  To trin-dummyer:
   step_2009 = I{year >= 2009} fanger regimeskiftet med G20 London/FATCA-
   forhåndsvarslet; step_2014 = I{year >= 2014} fanger
   FATCA-ikrafttrædelsesstødet.  Responsen er log(n), så
   koefficientestimater læses som proportionale effekter. */
data s18_ts;
    sæt year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
kør;

procedure udskriv data=s18_ts;
    titel "Section 18 — annual new-entity formations + intervention dummies";
kør;

procedure sgplot data=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x mærkat="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x mærkat="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis mærkat="Incorporation year";
    yaxis mærkat="New entities per year";
    titel "Section 18 — Paradise-Papers annual formations, 1970-2014";
kør;

/* Identificér modellen, og estimér derefter ARIMA(1,0,0) med de to trin-
   inputs.  PROC ARIMA's CROSSCORR= registrerer de eksogene variabler
   ved IDENTIFY-trinnet, så de er tilgængelige for ESTIMATE INPUT=. */
procedure arima data=s18_ts;
    identify variabel=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimate p=1 inddata=(step_2009 step_2014);
    titel "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
kør;
quit;


## 19. Zero-inflated antalsmodel for sanktionseksponering

**Reference:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2. udgave, Cambridge University Press (2013), kapitel 4.
Zero-inflated-modeller: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

Afsnit 14 fandt kun **fem** Paradise-Papers-selskaber med mindst
én officer på en konsolideret sanktionsliste — en forsvindende
sjælden begivenhed. En standard Poisson- eller negativ-binomial-regression på
`sanctioned_count` pr. selskab ville tilpasse dårligt, fordi **99,98 %** af
selskaberne har nul. Den zero-inflated negative-binomial (ZINB)-
model håndterer dette ved at opdele tilpasningen i to dele:

1. En logistisk model for, om selskabet tilhører en "strukturel
   nul"-klasse (ingen mulig sanktionseksponering).
2. En negativ-binomial-model for antallet blandt resten.

Med kun 5 positive begivenheder på tværs af 21.538 selskaber er ZINB-
likelihooden degenereret — begge dele kollapser. Den fejl er en
**ærlig egenskab ved dataene**, ikke ved proceduren. Vi kører
ZINB-tilpasningen alligevel for at vise, at regressionsværktøjet virker fra ende til anden,
og falder derefter tilbage til en almindelig binær-logistisk regression på
`has_sanctioned` (indikator for `sanctioned_count > 0`). Den
logistiske identificerer en ren effekt: **hver ekstra log-enhed af
officer-antal multiplicerer oddsene for at have mindst én
sanktioneret officer med omtrent 3,1** (p = 0,002).

Kovariater:

- `top5` — klassevariabel med 6 niveauer (BM, KY, VG, IM, JE, OTHER),
  referencekategori OTHER.
- `log_n_off` — `log(officer_count)`, den dominerende størrelsesprædiktor.


In [ ]:
/* Hent antal sanktionerede officerer pr. selskab fra den live graf. */
procedure gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

data s19;
    sæt s19_raw;
    hvis officer_count >= 1;
    længde top5 $5;
    top5 = 'OTHER';
    hvis jurisdiction = 'BM' så top5 = 'BM';
    ellers hvis jurisdiction = 'KY' så top5 = 'KY';
    ellers hvis jurisdiction = 'VG' så top5 = 'VG';
    ellers hvis jurisdiction = 'IM' så top5 = 'IM';
    ellers hvis jurisdiction = 'JE' så top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
kør;

procedure frekvenser data=s19;
    tables top5 sanctioned_count has_sanctioned;
    titel "Section 19 — response-variable distribution (very zero-heavy)";
kør;

/* ZINB først — forventes at være degenereret på en 5-begivenheds-serie. */
procedure genmod data=s19;
    klasse top5;
    model sanctioned_count = top5 log_n_off / dist=zinb link=log;
    titel "Section 19 — ZINB count model (degenerate on 5 events)";
kør;

/* Fallback: binær logistisk på has_sanctioned.  Fortolkelig. */
procedure logistic data=s19 faldende plots=none;
    klasse top5;
    model has_sanctioned = top5 log_n_off;
    titel "Section 19 — logistic regression (has-sanctioned fallback)";
kør;


## 20. Mixed-effects Poisson-model for stiftelsesrate

**Reference:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Klassisk panel-count: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

Afsnit 18 tilpassede en univariat ARIMA til den aggregerede stiftelsesserie.
Her bruger vi **panel**-dimensionen: én række pr. jurisdiktion-år-
celle, og tilpasser en Poisson generaliseret lineær mixed model (GLMM) med
en fast lineær årstendens plus en FATCA-trin-dummy, og et **tilfældigt
intercept pr. jurisdiktion**. Dette adskiller den fælles tendens-
effekt fra den enkelte jurisdiktions niveau.

Panelbegrænsning: de 10 jurisdiktioner, hvis **gennemsnitlige årlige
antal** er >=5 over 1990-2014 (203 celler i alt). Mindre
jurisdiktioner med mange nul-antals-år ville destabilisere en
Poisson-tilpasning.

`PROC GLIMMIX` med `DIST=POISSON LINK=LOG` og
`RANDOM INTERCEPT / SUBJECT=jurisdiction` producerer både de
populationsniveau-faste effekter (årstendens, FATCA-skift) og
variationskomponenten mellem jurisdiktioner. Variansen for det tilfældige
intercept fortæller os, *hvor meget stiftelsesrater varierer på tværs af
jurisdiktioner efter hensyntagen til den fælles tidstendens* — en
strukturel heterogenitetsscore for populationen af offshore-finanscentre.


In [ ]:
/* Paneldatasæt: jurisdiktion x år-celler fra 1990-2014. */
procedure gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Behold jurisdiktioner med gennemsnitligt-årligt-antal >= 5. */
procedure sql;
    create table s20_keep_jur as
    vælg jurisdiction, avg(n) as avg_n
    from s20_raw
    group efter jurisdiction
    having avg(n) >= 5;
quit;

procedure sql;
    create table s20 as
    vælg r.*,
           r.year - 2002 as year_c,
           case når r.year >= 2014 så 1 ellers 0 slut as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

procedure frekvenser data=s20;
    tables jurisdiction;
    titel "Section 20 — jurisdictions retained in the panel";
kør;

/* Mixed-effects Poisson-GLMM: fast årstendens + FATCA-trin,
   tilfældigt intercept pr. jurisdiktion. */
procedure glimmix data=s20;
    klasse jurisdiction;
    model n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    titel "Section 20 — mixed Poisson formation-rate model";
kør;

/* Rangerede tilfældige-intercept-effekter ville fremhæve de
   jurisdiktioner, der systematisk over- eller underpræsterer i forhold til
   den fælles tendens.  PROC GLIMMIX SOLUTION-sætningen udskriver disse
   i standard-outputtet ovenfor — vi overlader rangeringen til
   læseren. */


In [ ]:
/* Luk rapport-PDF'en og frigiv Neo4j-biblioteket. */
ods pdf close;

libname icij clear;


## Reproducerbarhed og proveniens

- **Grafens datakilde:** ICIJ *Offshore Leaks*-forskningsdatabase,
  *Paradise Papers*-datasæt, første gang udgivet i november 2017.
  Tilgængelig på <https://offshoreleaks.icij.org/pages/database>.
  Indlæst i platformens delte `step-neo4j`-tjeneste
  (Neo4j 5.26 Community Edition, skrivebeskyttet på serverniveau) via det
  offentlige upstream-dump på
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **OFAC-SDN-data:** USA's Treasury OFAC Specially Designated
  Nationals offentlige CSV-eksport, hentet fra Treasurys offentlige
  preview-API i april 2026. Filen `data/ofac_sdn.csv` indeholder
  500 kuraterede rækker på tværs af de fem største OFAC-programmer efter
  antal poster. Brugt til totrins-screeningen i afsnit 6.
- **OpenSanctions-data:** OpenSanctions *default*-konsoliderede
  datasæt-snapshot fra 2026-04-17, downloadet fra
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Den lagrede fil `data/opensanctions_default.csv` indeholder de
  18.654 rækker med Person- og Company-skema, hvis primære datasæt er
  ét af sanktionslisterne fra OFAC SDN, UK HM Treasury, EU's finansielle sanktioner, FN's
  Sikkerhedsråd, eller de canadiske, belgiske, australske, schweiziske eller andre
  nationale konsoliderede sanktionslister. Aliasser blev droppet for at
  holde filen under 2 MB. Licens: ODbL 1.0 (OpenSanctions). Brugt
  til beriggelsen i afsnit 14.
- **Skattely-placeringer:** Tax Justice Networks *Corporate Tax Haven
  Index 2021*-offentliggjorte placeringer, sammensat fra
  `https://cthi.taxjustice.net`-indeksets landingsside og pressemeddelelsen fra
  marts 2021 på
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Den lagrede fil `data/tax_haven_ranks.csv` opregner de
  jurisdiktioner, der optræder i Paradise Papers, med deres
  CTHI-placering og en afledt risikovægt i `[0, 1]`. Licens: CC BY-SA
  4.0 (Tax Justice Network). Brugt til den sammensatte
  rangering i afsnit 15.
- **Graf-algoritmer:** Louvain-fællesskabsdetektion og
  egenvektorcentralitet (den nærmeste interne analog til PageRank)
  beregnet af `PROC NETWORK` in-process på kantlister hentet via
  skrivebeskyttet Cypher. Platformens delte `step-neo4j`-tjeneste er
  skrivebeskyttet på serverniveau, så alle graf-algoritmer kører i
  workspace-pod'en frem for via skriveoperationer i Neo4j Graph Data
  Science.
- **Statistiske metoder:** `PROC LIFETEST` bruger Kaplan-Meier-
  estimatoren med stratificerede log-rank-, Wilcoxon- og Tarone-Ware-
  tests. `PROC PHREG` tilpasser Cox proportional-hazards-modellen via
  Breslow-ties ved hjælp af Python/statsmodels-wrapperen. `PROC LOGISTIC`
  tilpasser en maximum-likelihood binær logistisk regression.
- **Metoder til risikosammensætning:** Den sammensatte indflydelses-
  score i afsnit 11 normaliserer grad, log-PageRank, jurisdiktionel bredde
  og anciennitetsår til `[0, 1]` og summerer med faste vægte
  (30/30/20/20). Den sammensatte risikoscore for selskaber i afsnit 15
  normaliserer loftlagt officer-antal, log-PageRank, CTHI-risikovægt
  og et binært flag for sanktioneret officer og summerer med lige vægte
  på 0,25 hver.

Den komplette analyse er reproducerbar fra smoke-test-scriptet
i denne mappe: `./smoke.jenner`. At køre det fra ende til anden mod den
delte `step-neo4j`-tjeneste (med `JENNER_NEO4J_HOST` og
`JENNER_NEO4J_PASS` sat, hvilket platformen gør for dig i enhver
workspace-pod) tager omtrent to minutter og verificerer, at hver
forespørgsel og hver PROC — inklusive de fem nye afsnit tilføjet
sammen med den eksisterende pipeline — returnerer det forventede output på
den reelle graf med 163.435 knuder.
